# ColabUMA — UMAによる分子構造最適化

Metaの機械学習ポテンシャル **UMA** で分子の構造最適化を行うノートブックです。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で SMILES を入れるか xyz アップロードを選び、電荷とスピン多重度を指定する
4. **ランタイム → すべて実行** を押す（xyzアップロードを選んだ場合は、途中でファイル選択ボタンが出ます）
5. 一番下のセルで 3D 表示・エネルギー・収束グラフを確認し、結果のZIPをダウンロードする

2回目以降は手順3〜5を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- SMILESから作った初期構造は電荷やスピンを考慮していないため、鞍点に収束することがあります。重要な構造は **振動計算** をオンにして確認してください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 小さな分子（数十原子以下）では、DFTを直接使う方が確実な場面が多くあります。
- 遷移状態探索・IRC・周期系（結晶・表面）の最適化には対応していません。

## ライセンス

- このノートのコード：MIT License
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。

#@markdown ### 構造
input_mode = "SMILES"  #@param ["SMILES", "xyzアップロード"]
smiles = "CCO"  #@param {type:"string"}
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
fmax = 0.05  #@param {type:"number"}
max_steps = 500  #@param {type:"integer"}
n_conformers = 1  #@param {type:"integer"}
run_vibrations = False  #@param {type:"boolean"}

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

In [ ]:
#@title 2. セットアップ（GPU確認・インストール・認証） { display-mode: "form" }
FAIRCHEM_VERSION = "2.25.0"

import html
import os
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as _pkg_version

from IPython import get_ipython
from IPython.display import HTML, display


class ColabUMAError(Exception):
    # 利用者に見せるエラー。トレースバックの代わりに日本語の一文と対処法を表示する
    def __init__(self, message, hint=""):
        super().__init__(message)
        self.message = message
        self.hint = hint


def _box(kind, message, hint=""):
    bg, fg, label = {
        "error": ("#fdecea", "#b3261e", "エラー"),
        "warn": ("#fff4e5", "#8a5300", "注意"),
        "ok": ("#e8f5e9", "#1b5e20", "OK"),
    }[kind]
    hint_html = f"<div style='margin-top:4px'>対処法：{html.escape(hint)}</div>" if hint else ""
    display(HTML(
        f"<div style='background:{bg};color:{fg};padding:10px 14px;border-radius:6px;"
        f"margin:6px 0;font-size:14px;line-height:1.6'><b>{label}</b>　{html.escape(message)}{hint_html}</div>"
    ))


def warn(message, hint=""):
    _box("warn", message, hint)


def ok(message):
    _box("ok", message)


def _show_colabuma_error(shell, etype, evalue, tb, tb_offset=None):
    _box("error", evalue.message, evalue.hint)
    return []


get_ipython().set_custom_exc((ColabUMAError,), _show_colabuma_error)


def _installed(pkg):
    try:
        return _pkg_version(pkg)
    except PackageNotFoundError:
        return None


# --- GPUの確認（torchを読み込む前に nvidia-smi で調べる） ---
_has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
if not _has_gpu:
    warn("GPUが見つからないため、CPUで計算します。計算はかなり遅くなります。",
         "メニューの「ランタイム → ランタイムのタイプを変更」で「T4 GPU」を選び、もう一度すべて実行してください。")

# --- ライブラリのインストール（インストール済みならスキップ） ---
if _installed("fairchem-core") != FAIRCHEM_VERSION or not _installed("rdkit") or not _installed("py3Dmol"):
    _torch_loaded_before = "torch" in sys.modules
    print(f"ライブラリをインストールしています（fairchem-core {FAIRCHEM_VERSION}）。数分かかります…")
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"fairchem-core=={FAIRCHEM_VERSION}", "rdkit", "py3Dmol"],
        capture_output=True, text=True)
    if _r.returncode != 0:
        print("\n".join(_r.stderr.strip().splitlines()[-15:]))
        raise ColabUMAError("ライブラリのインストールに失敗しました（上にpipのメッセージを表示しています）。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    if _torch_loaded_before:
        raise ColabUMAError("インストールは完了しましたが、読み込み済みのライブラリを入れ替えるためセッションの再起動が必要です。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    ok("インストールが完了しました。")
else:
    print(f"fairchem-core {FAIRCHEM_VERSION} はインストール済みです。スキップします。")

# --- Hugging Faceの認証（トークンはノートに保存しない） ---
if not globals().get("_COLABUMA_AUTH_OK"):
    _token = os.environ.get("HF_TOKEN")
    if not _token:
        try:
            from google.colab import userdata
            _token = userdata.get("HF_TOKEN")
        except Exception:
            _token = None
    if not _token:
        warn("ColabのシークレットにHF_TOKENが登録されていません。",
             "左サイドバーの鍵アイコン（シークレット）で、名前 HF_TOKEN・値にトークンを登録し、"
             "「ノートブックからのアクセス」をオンにしてください。今回は下の欄に貼り付けても続行できます。")
        from getpass import getpass
        _token = getpass("Hugging Faceのアクセストークンを貼り付けてEnter: ").strip()
    if not _token:
        raise ColabUMAError("Hugging Faceのアクセストークンがありません。",
                            "https://huggingface.co/settings/tokens でRead権限のトークンを作り、シークレットにHF_TOKENとして登録してください。")
    os.environ["HF_TOKEN"] = _token
    del _token

    from huggingface_hub import whoami
    from huggingface_hub.utils import GatedRepoError, HfHubHTTPError, RepositoryNotFoundError
    try:
        whoami()
    except Exception as e:
        if getattr(getattr(e, "response", None), "status_code", None) == 401:
            raise ColabUMAError("Hugging Faceのトークンが無効です。",
                                "https://huggingface.co/settings/tokens でトークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    try:
        from huggingface_hub import auth_check
        auth_check("facebook/UMA")
    except ImportError:
        pass  # 古いhuggingface_hubでは確認を省略し、モデル読み込み時に判定する
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except RepositoryNotFoundError:
        raise ColabUMAError("Hugging Faceのトークンが無効です。",
                            "トークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
    except HfHubHTTPError as e:
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    _COLABUMA_AUTH_OK = True
    ok("Hugging Faceの認証とfacebook/UMAへのアクセス権を確認しました。")

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("計算デバイス:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

In [ ]:
#@title 3. モデル読み込み { display-mode: "form" }
import gc

from fairchem.core import FAIRChemCalculator, pretrained_mlip
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

_key = (model_name, DEVICE)
if globals().get("_UMA_KEY") != _key:
    globals().pop("_UMA_PREDICTOR", None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{model_name} を読み込んでいます（初回はダウンロードに数分かかります）…")
    try:
        _UMA_PREDICTOR = pretrained_mlip.get_predict_unit(model_name, device=DEVICE)
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except HfHubHTTPError as e:
        _status = getattr(getattr(e, "response", None), "status_code", None)
        if _status in (401, 403):
            raise ColabUMAError("モデルのダウンロードが拒否されました（アクセス権がありません）。",
                                "https://huggingface.co/facebook/UMA でアクセス申請し、トークンが正しいか確認してください。") from None
        raise ColabUMAError(f"モデルをダウンロードできませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    except KeyError:
        raise ColabUMAError(f"モデル名「{model_name}」はこのバージョンのfairchemにありません。",
                            "フォームで別のモデルを選んでください。") from None
    _UMA_KEY = _key
    ok(f"{model_name} を読み込みました。")
else:
    print(f"{model_name} は読み込み済みです。スキップします。")

calc = FAIRChemCalculator(_UMA_PREDICTOR, task_name=task_name)

In [ ]:
#@title 4. 構造の準備 { display-mode: "form" }
import re
import tempfile

import numpy as np
from ase import Atoms
from ase.data import chemical_symbols
from ase.io import read as ase_read

MAX_Z = 83  # UMAのomolタスク（OMol25）が扱える元素は H〜Bi

# --- フォームの値の確認 ---
if fmax <= 0:
    raise ColabUMAError("収束判定 fmax は正の数にしてください。", "通常は 0.05 eV/Å 程度です。")
if max_steps < 1:
    raise ColabUMAError("最大ステップ数は1以上にしてください。", "通常は 500 程度です。")
if n_conformers < 1:
    raise ColabUMAError("配座数は1以上にしてください。", "1なら配座探索をせずに1つだけ最適化します。")
if multiplicity < 1:
    raise ColabUMAError("スピン多重度は1以上の整数にしてください。", "閉殻分子なら1（一重項）です。")
out_prefix = re.sub(r"[^\w\-.]", "_", output_name.strip()) or "result"


def check_elements(numbers):
    bad = sorted({chemical_symbols[z] if 0 < z < len(chemical_symbols) else "X"
                  for z in numbers if z < 1 or z > MAX_Z})
    if bad:
        raise ColabUMAError(f"UMAが扱えない元素が含まれています：{', '.join(bad)}",
                            "UMA（omol）が扱えるのは H〜Bi（原子番号1〜83）です。該当する元素を除くか、別の手法を使ってください。")


# --- 初期構造の作成 ---
if input_mode == "SMILES":
    from rdkit import Chem, RDLogger
    from rdkit.Chem import AllChem

    RDLogger.DisableLog("rdApp.*")
    mol = Chem.MolFromSmiles(smiles.strip()) if smiles.strip() else None
    if mol is None:
        raise ColabUMAError(f"SMILES「{smiles}」を読み取れませんでした。",
                            "括弧や環番号の対応、原子記号の大文字・小文字、芳香族の書き方を見直してください（例：エタノールは CCO）。")
    if Chem.GetFormalCharge(mol) != charge:
        warn(f"SMILESの形式電荷（{Chem.GetFormalCharge(mol):+d}）とフォームの電荷（{charge:+d}）が違います。計算にはフォームの電荷を使います。",
             "意図した電荷か確認してください。")
    check_elements([a.GetAtomicNum() for a in mol.GetAtoms()])
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params))
    if not cids:
        params.useRandomCoords = True
        cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params))
    if not cids:
        raise ColabUMAError("SMILESから3D構造を生成できませんでした。", "SMILESを見直すか、xyzファイルで入力してください。")
    if len(cids) < n_conformers:
        warn(f"配座は {n_conformers} 個のうち {len(cids)} 個しか生成できませんでした。生成できた分だけ計算します。")
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000)
    else:
        warn("MMFFのパラメータがない元素を含むため、MMFFによる前処理を省略しました。")
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    conformers = [Atoms(symbols, positions=mol.GetConformer(c).GetPositions()) for c in cids]
    source_desc = f"SMILES: {smiles.strip()}"
else:
    from google.colab import files

    print("xyzファイルを選んでください。")
    uploaded = files.upload()
    if not uploaded:
        raise ColabUMAError("ファイルがアップロードされませんでした。", "このセルをもう一度実行して、xyzファイルを選んでください。")
    if len(uploaded) > 1:
        warn("複数のファイルが選ばれたため、最初の1つだけを使います。")
    fname, data = next(iter(uploaded.items()))
    path = os.path.join(tempfile.mkdtemp(), os.path.basename(fname))
    with open(path, "wb") as f:
        f.write(data)
    frames = None
    for fmt in ("extxyz", "xyz"):
        try:
            frames = ase_read(path, index=":", format=fmt)
            break
        except Exception:
            pass
    if not frames:
        raise ColabUMAError(f"「{fname}」をxyzファイルとして読み込めませんでした。",
                            "1行目が原子数、2行目がコメント、3行目以降が「元素記号 x y z」（単位Å）の形式か確認してください。")
    if len(frames) > 1:
        warn(f"「{fname}」には {len(frames)} 個の構造が入っています。先頭の構造だけを使います。")
    atoms0 = frames[0]
    atoms0.set_pbc(False)
    atoms0.set_cell(np.zeros((3, 3)))
    atoms0.info = {}
    conformers = [atoms0]
    if n_conformers > 1:
        warn("配座数はSMILES入力のときだけ使われます。今回はアップロードした構造を1つ最適化します。")
    source_desc = f"xyz: {fname}"

# --- 元素・電荷・多重度の確認 ---
numbers = conformers[0].numbers
check_elements(numbers)
n_electrons = int(numbers.sum()) - int(charge)
n_unpaired = int(multiplicity) - 1
if n_electrons <= 0 or n_unpaired > n_electrons or (n_electrons - n_unpaired) % 2:
    suggest = "1, 3, …" if n_electrons % 2 == 0 else "2, 4, …"
    raise ColabUMAError(
        f"電荷 {charge:+d}・スピン多重度 {multiplicity} の組み合わせはこの分子では成り立ちません（電子数 {n_electrons}）。",
        f"多重度は 2S+1 です。電子数が偶数なら多重度は奇数、奇数なら偶数になります。この電荷なら多重度は {suggest} が候補です。")
for a in conformers:
    a.info.update({"charge": int(charge), "spin": int(multiplicity)})

ok(f"{conformers[0].get_chemical_formula()}（{len(conformers[0])} 原子）の初期構造を {len(conformers)} 個用意しました。")

In [ ]:
#@title 5. 最適化 { display-mode: "form" }
import time

from ase.optimize import BFGS, FIRE, LBFGS

OPTIMIZERS = {"LBFGS": LBFGS, "BFGS": BFGS, "FIRE": FIRE}
IMAG_THRESHOLD = 20.0  # cm^-1。これより小さい虚振動は数値誤差とみなす


def _fmax_of(forces):
    return float(np.sqrt((forces ** 2).sum(axis=1).max()))


def optimize(atoms, label):
    atoms = atoms.copy()
    atoms.calc = calc
    hist = {"energy": [], "fmax": [], "frames": []}
    opt = OPTIMIZERS[optimizer_name](atoms, logfile=None)

    def record():
        e = float(atoms.get_potential_energy())
        fm = _fmax_of(atoms.get_forces())
        hist["energy"].append(e)
        hist["fmax"].append(fm)
        hist["frames"].append(atoms.positions.copy())
        n = len(hist["energy"]) - 1
        if n % 10 == 0:
            print(f"\r  {label}: step {n:4d}  E = {e:.6f} eV  fmax = {fm:.4f} eV/Å", end="")

    opt.attach(record, interval=1)
    t0 = time.time()
    try:
        opt.run(fmax=fmax, steps=max_steps)
    except torch.cuda.OutOfMemoryError:
        raise ColabUMAError("GPUのメモリが足りません。",
                            "モデルを uma-s-1p1 などの小さいものにするか、原子数を減らしてください。") from None
    converged = hist["fmax"][-1] < fmax
    print(f"\r  {label}: step {opt.nsteps:4d}  E = {hist['energy'][-1]:.6f} eV  fmax = {hist['fmax'][-1]:.4f} eV/Å"
          f"  {'収束' if converged else '未収束'}（{time.time() - t0:.1f} 秒）")
    return {"label": label, "atoms": atoms, "energy": hist["energy"][-1], "fmax": hist["fmax"][-1],
            "steps": opt.nsteps, "converged": converged, "hist": hist}


print(f"{optimizer_name} で最適化します（fmax < {fmax} eV/Å、最大 {max_steps} ステップ）")
results = [optimize(a, f"配座 {i + 1}" if len(conformers) > 1 else "構造") for i, a in enumerate(conformers)]
best = min(results, key=lambda r: r["energy"])
best_atoms = best["atoms"]

# --- 振動計算（オンのときだけ） ---
vib_result = None
if run_vibrations:
    if len(best_atoms) == 1:
        warn("原子が1つなので振動計算は行いません。")
    else:
        from ase.vibrations import Vibrations

        print("振動計算をしています…")
        va = best_atoms.copy()
        va.calc = calc
        vib = Vibrations(va, name=os.path.join(tempfile.mkdtemp(), "vib"), delta=0.01)
        vib.run()
        freqs = vib.get_frequencies()
        # 虚振動は負の値で表す
        signed = np.where(np.abs(freqs.imag) > np.abs(freqs.real), -np.abs(freqs.imag), freqs.real)
        inertia = va.get_moments_of_inertia()
        n_ext = 5 if inertia.min() < 1e-3 * inertia.max() else 6
        internal = np.sort(signed[np.argsort(np.abs(signed))][n_ext:])
        n_imag = int((internal < -IMAG_THRESHOLD).sum())
        vib_result = {"all": np.sort(signed), "internal": internal, "n_imag": n_imag, "n_ext": n_ext}

In [ ]:
#@title 6. 結果表示と保存 { display-mode: "form" }
import datetime
import platform
import zipfile

import ase
import ase.units
import matplotlib.pyplot as plt
import pandas as pd
import py3Dmol

EV_PER_HARTREE = ase.units.Hartree
EV_PER_KCALMOL = ase.units.kcal / ase.units.mol


def xyz_block(symbols, positions, comment):
    lines = [str(len(symbols)), comment]
    lines += [f"{s:<2s} {x:16.10f} {y:16.10f} {z:16.10f}" for s, (x, y, z) in zip(symbols, positions)]
    return "\n".join(lines) + "\n"


symbols = best_atoms.get_chemical_symbols()
E, E_h = best["energy"], best["energy"] / EV_PER_HARTREE
cond = (f"model={model_name} task={task_name} charge={charge} multiplicity={multiplicity} "
        f"optimizer={optimizer_name} fmax_target={fmax}")
opt_xyz = xyz_block(symbols, best_atoms.positions,
                    f"energy_eV={E:.8f} energy_Ha={E_h:.10f} fmax={best['fmax']:.5f} "
                    f"converged={best['converged']} {cond}")

# --- 3D表示 ---
view = py3Dmol.view(width=640, height=420)
view.addModel(opt_xyz, "xyz")
view.setStyle({"stick": {"radius": 0.14}, "sphere": {"scale": 0.24}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

# --- 数値のまとめ ---
summary = pd.DataFrame({"値": [
    f"{best_atoms.get_chemical_formula()}（{len(best_atoms)} 原子）",
    f"{E:.6f} eV",
    f"{E_h:.8f} Hartree",
    f"{best['fmax']:.4f} eV/Å",
    str(best["steps"]),
    "はい" if best["converged"] else "いいえ",
]}, index=["組成", "最終エネルギー", "最終エネルギー", "最終 fmax", "ステップ数", "収束"])
display(summary)
if not all(r["converged"] for r in results):
    warn(f"最大ステップ数（{max_steps}）までに収束しなかった構造があります。結果は途中の構造です。",
         "最大ステップ数を増やすか、fmax を緩めてもう一度実行してください。")

# --- 収束グラフ（Colabは日本語フォントがないため軸ラベルは英語） ---
h = best["hist"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(h["energy"], marker=".", color="#1f77b4")
ax1.set_xlabel("Step")
ax1.set_ylabel("Energy (eV)")
ax1.set_title("Energy")
ax2.semilogy(h["fmax"], marker=".", color="#d62728")
ax2.axhline(fmax, ls="--", color="gray", label=f"fmax = {fmax}")
ax2.set_xlabel("Step")
ax2.set_ylabel("Max force (eV/Å)")
ax2.set_title("Max force")
ax2.legend()
fig.tight_layout()
plt.show()

# --- 配座の比較 ---
conf_df = None
if len(results) > 1:
    conf_df = pd.DataFrame([{
        "配座": r["label"],
        "エネルギー (eV)": round(r["energy"], 6),
        "エネルギー (Hartree)": round(r["energy"] / EV_PER_HARTREE, 8),
        "ΔE (kcal/mol)": round((r["energy"] - best["energy"]) / EV_PER_KCALMOL, 3),
        "ステップ数": r["steps"],
        "収束": "はい" if r["converged"] else "いいえ",
        "最安定": "★" if r is best else "",
    } for r in results])
    display(conf_df.style.hide(axis="index"))

# --- 振動計算の結果 ---
if vib_result is not None:
    vmin = vib_result["internal"].min()
    print(f"虚振動の本数: {vib_result['n_imag']}（{IMAG_THRESHOLD:.0f} cm⁻¹ 未満の虚振動は数値誤差として除外）")
    print(f"最小振動数: {vmin:.1f} cm⁻¹（負の値は虚振動）")
    if vib_result["n_imag"] > 0:
        warn(f"虚振動が {vib_result['n_imag']} 本あります。極小点ではなく鞍点の可能性があります。",
             "構造を少しずらすか、配座数を増やして再最適化してください。")
    else:
        ok("虚振動はありません。極小点に収束しています。")

# --- 出力ファイルの作成 ---
out_dir = os.path.join(tempfile.mkdtemp(), out_prefix)
os.makedirs(out_dir)
written = {}


def save(name, text):
    p = os.path.join(out_dir, f"{out_prefix}_{name}")
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    written[name] = p


save("opt.xyz", opt_xyz)
save("traj.xyz", "".join(
    xyz_block(symbols, pos, f"step={i} energy_eV={e:.8f} fmax={fm:.5f}")
    for i, (pos, e, fm) in enumerate(zip(h["frames"], h["energy"], h["fmax"]))))


def _v(pkg):
    return _installed(pkg) or "-"


log = [
    "ColabUMA 計算ログ",
    f"日時: {datetime.datetime.now().isoformat(timespec='seconds')}",
    "",
    "[入力条件]",
    f"入力: {source_desc}",
    f"電荷: {charge}", f"スピン多重度: {multiplicity}",
    f"モデル: {model_name}", f"タスク: {task_name}",
    f"最適化アルゴリズム: {optimizer_name}", f"fmax: {fmax} eV/Å", f"最大ステップ数: {max_steps}",
    f"配座数: {n_conformers}（生成 {len(conformers)}）", f"振動計算: {'オン' if run_vibrations else 'オフ'}",
    "",
    "[ソフトウェア]",
    f"Python {platform.python_version()}", f"fairchem-core {_v('fairchem-core')}", f"ase {ase.__version__}",
    f"torch {torch.__version__}", f"rdkit {_v('rdkit')}", f"numpy {np.__version__}",
    f"デバイス: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}",
    "",
    "[結果]",
    f"最安定: {best['label']}  E = {E:.8f} eV ({E_h:.10f} Ha)  fmax = {best['fmax']:.5f}  "
    f"steps = {best['steps']}  converged = {best['converged']}",
]
for r in results:
    log += ["", f"[{r['label']}] steps = {r['steps']}  converged = {r['converged']}",
            f"{'step':>5s} {'energy_eV':>18s} {'fmax_eV/A':>12s}"]
    log += [f"{i:5d} {e:18.8f} {fm:12.5f}" for i, (e, fm) in enumerate(zip(r["hist"]["energy"], r["hist"]["fmax"]))]
save("log.txt", "\n".join(log) + "\n")

if conf_df is not None:
    save("conformers.csv", conf_df.to_csv(index=False))
if vib_result is not None:
    freq_lines = [f"# 振動数 (cm^-1)。負の値は虚振動。並進・回転 {vib_result['n_ext']} 個を除いた値",
                  f"# 虚振動の本数（|ν| >= {IMAG_THRESHOLD:.0f} cm^-1）: {vib_result['n_imag']}"]
    freq_lines += [f"{i + 1:4d} {v:12.2f}" for i, v in enumerate(vib_result["internal"])]
    freq_lines += ["", "# 全モード（並進・回転を含む）"] + [f"{i + 1:4d} {v:12.2f}" for i, v in enumerate(vib_result["all"])]
    save("freq.txt", "\n".join(freq_lines) + "\n")

zip_path = os.path.join(os.path.dirname(out_dir), f"{out_prefix}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written.values():
        z.write(p, os.path.basename(p))
print("ZIPの中身:", ", ".join(os.path.basename(p) for p in written.values()))

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("ZIPを保存しました:", zip_path)